In [42]:
import pandas as pd

# Load raw branches data
branches = pd.read_csv("../data/raw/branches.csv")

branches.head()


,branch_id,branch_name,city,state,region,warehouse_type,warehouse_capacity,service_center_available,manager_id,total_employees,avg_monthly_revenue,monthly_operational_cost,market_demand_index
0,DEL001,Delhi Central,Delhi,Delhi,North,Central,45230 sqft,Yes,9593,92,5234890,3189000,9
1,PUN001,Pune Distribution,Pune,Maharashtra,West,Regional,38750 sqft,Yes,6114,74,4389120,2654000,8
2,CHN001,Chennai South Hub,Chennai,Tamil Nadu,South,Regional,41890 sqft,Yes,7922,81,4567780,2739000,8
3,HYD001,Hyderabad Logistics,Hyderabad,Telangana,South,Branch,36210 sqft,No,4390,57,3102455,1887000,7
4,KOL001,Kolkata East Depot,Kolkata,West Bengal,East,Branch,29880 sqft,No,5981,49,2894770,1743000,6


In [ ]:
print("=== INFO ===")
print(branches.info())

print("\n=== MISSING VALUES ===")
print(branches.isnull().sum())

print("\n=== SAMPLE DATA ===")
branches.head()


=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   branch_id                 6 non-null      str  
 1   branch_name               6 non-null      str  
 2   city                      6 non-null      str  
 3   state                     6 non-null      str  
 4   region                    6 non-null      str  
 5   warehouse_type            6 non-null      str  
 6   warehouse_capacity        6 non-null      str  
 7   service_center_available  6 non-null      str  
 8   manager_id                6 non-null      int64
 9   total_employees           6 non-null      int64
 10  avg_monthly_revenue       6 non-null      int64
 11  monthly_operational_cost  6 non-null      int64
 12  market_demand_index       6 non-null      int64
dtypes: int64(5), str(8)
memory usage: 756.0 bytes
None

=== MISSING VALUES ===
branch_id             

,branch_id,branch_name,city,state,region,warehouse_type,warehouse_capacity,service_center_available,manager_id,total_employees,avg_monthly_revenue,monthly_operational_cost,market_demand_index
0,DEL001,Delhi Central,Delhi,Delhi,North,Central,45230 sqft,Yes,9593,92,5234890,3189000,9
1,PUN001,Pune Distribution,Pune,Maharashtra,West,Regional,38750 sqft,Yes,6114,74,4389120,2654000,8
2,CHN001,Chennai South Hub,Chennai,Tamil Nadu,South,Regional,41890 sqft,Yes,7922,81,4567780,2739000,8
3,HYD001,Hyderabad Logistics,Hyderabad,Telangana,South,Branch,36210 sqft,No,4390,57,3102455,1887000,7
4,KOL001,Kolkata East Depot,Kolkata,West Bengal,East,Branch,29880 sqft,No,5981,49,2894770,1743000,6


In [44]:
# Convert warehouse_capacity ("45230 sqft") → integer
branches['warehouse_capacity'] = (
    branches['warehouse_capacity']
    .astype(str)                         # ensure everything is a string
    .str.replace(' sqft', '', regex=False)
    .astype(int)
)


# Convert Yes/No → boolean (1/0)
branches['service_center_available'] = branches['service_center_available'].map({'Yes': 1, 'No': 0})


# branches['service_center_available'] = (
#     branches['service_center_available'].map({'Yes': 1, 'No': 0})
# )
# Convert numeric columns stored as strings
numeric_cols = [
    'employee_count', 'vehicle_count',
    'annual_operating_cost', 'annual_revenue',
    'rating'
]

for col in numeric_cols:
    if col in branches.columns:
        branches[col] = pd.to_numeric(branches[col], errors='coerce')


In [45]:
# Fill numeric columns with median
for col in numeric_cols:
    if col in branches.columns:
        branches[col] = branches[col].fillna(branches[col].median())

# Fill categorical columns with mode
if 'warehouse_type' in branches.columns:
    branches['warehouse_type'] = branches['warehouse_type'].fillna(branches['warehouse_type'].mode()[0])
if 'region' in branches.columns:
    branches['region'] = branches['region'].fillna(branches['region'].mode()[0])


In [46]:
if 'region' in branches.columns:
    branches['region'] = branches['region'].str.title()
if 'warehouse_type' in branches.columns:
    branches['warehouse_type'] = branches['warehouse_type'].str.title()
if 'city' in branches.columns:
    branches['city'] = branches['city'].str.title()
if 'state' in branches.columns:
    branches['state'] = branches['state'].str.title()

In [47]:
branches.drop_duplicates(inplace=True)


In [48]:
# Create derived metrics only when the required source columns exist.
# Some versions of this dataset use avg_monthly_revenue/monthly_operational_cost
# instead of annual_revenue/annual_operating_cost, and may not contain current_stock.
if 'warehouse_capacity' in branches.columns:
    if 'current_stock' in branches.columns:
        branches['utilization_rate'] = branches['current_stock'] / branches['warehouse_capacity']
    else:
        branches['utilization_rate'] = pd.NA

    revenue_col = 'annual_revenue' if 'annual_revenue' in branches.columns else 'avg_monthly_revenue'
    if revenue_col in branches.columns:
        branches['revenue_per_sqft'] = branches[revenue_col] / branches['warehouse_capacity']
    else:
        branches['revenue_per_sqft'] = pd.NA

    if 'annual_revenue' in branches.columns:
        cost_col = 'annual_operating_cost' if 'annual_operating_cost' in branches.columns else 'monthly_operational_cost'
        if cost_col in branches.columns:
            branches['cost_revenue_ratio'] = branches[cost_col] / branches['annual_revenue']
        else:
            branches['cost_revenue_ratio'] = pd.NA
    elif 'avg_monthly_revenue' in branches.columns and 'monthly_operational_cost' in branches.columns:
        branches['cost_revenue_ratio'] = branches['monthly_operational_cost'] / branches['avg_monthly_revenue']
    else:
        branches['cost_revenue_ratio'] = pd.NA


In [49]:
print("=== DUPLICATE BRANCH IDs ===")
print(branches['branch_id'].duplicated().sum())

print("\n=== REGION VALUES ===")
print(branches['region'].unique())

print("\n=== NUMERIC SUMMARY ===")
available_numeric_cols = [
	column for column in numeric_cols
	if column in branches.columns
]

if available_numeric_cols:
	print(branches[available_numeric_cols].describe())
else:
	print("No columns from numeric_cols are present in branches.")


=== DUPLICATE BRANCH IDs ===
0

=== REGION VALUES ===
<StringArray>
['North', 'West', 'South', 'East']
Length: 4, dtype: str

=== NUMERIC SUMMARY ===
No columns from numeric_cols are present in branches.


In [50]:
branches.to_csv('../data/processed/branches.csv', index=False)

print("Processed branches.csv saved to data/processed/")


Processed branches.csv saved to data/processed/
